In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("aryansraut/preprocessed-ucf-crime-dataset-visual")

print("Path to dataset files:", path)

Mounting files to /kaggle/input/datasets/aryansraut/preprocessed-ucf-crime-dataset-visual...
Path to dataset files: /kaggle/input/datasets/aryansraut/preprocessed-ucf-crime-dataset-visual


In [2]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models.video import r3d_18
from PIL import Image
import numpy as np
from tqdm import tqdm
from sklearn.metrics import classification_report

In [3]:
class TikHarmVideoDataset(Dataset):
    def __init__(self, root_dir, split, transform=None):
        self.samples = []
        self.transform = transform
        
        classes = ["Adult Content", "Harmful Content", "Safe", "Suicide"]
        self.class_to_idx = {c: i for i, c in enumerate(classes)}

        for cls in classes:
            cls_path = os.path.join(root_dir, split, cls)
            for video_id in os.listdir(cls_path):
                self.samples.append(
                    (os.path.join(cls_path, video_id), self.class_to_idx[cls])
                )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        frame_dir, label = self.samples[idx]
        frames = sorted(os.listdir(frame_dir))[:16]

        imgs = []
        for f in frames:
            img = Image.open(os.path.join(frame_dir, f)).convert("RGB")
            if self.transform:
                img = self.transform(img)
            imgs.append(img)

        video = torch.stack(imgs)          # [T, C, H, W]
        video = video.permute(1, 0, 2, 3)  # [C, T, H, W]
        return video, label

In [17]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.43216, 0.394666, 0.37645],
        std=[0.22803, 0.22145, 0.216989]
    )
])

In [18]:
root = "/kaggle/input/datasets/aryansraut/preprocessed-ucf-crime-dataset-visual/TikHarm_frames_16"
train_ds = TikHarmVideoDataset(root, "train", transform)
val_ds   = TikHarmVideoDataset(root, "val", transform)
test_ds  = TikHarmVideoDataset(root, "test", transform)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=4, pin_memory=True)

print(len(train_ds), len(val_ds), len(test_ds))

2762 396 790


In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = r3d_18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 4)

model = model.to(device)

if torch.cuda.device_count() > 1:
    print("Using", torch.cuda.device_count(), "GPUs")
    model = nn.DataParallel(model)

Using 2 GPUs


In [22]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=2
)

In [11]:
def train_epoch(model, loader):
    model.train()
    total_loss = 0
    
    for videos, labels in tqdm(loader):
        videos, labels = videos.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(videos)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    return total_loss / len(loader)

In [24]:
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for videos, labels in loader:
            videos, labels = videos.to(device), labels.to(device)
            outputs = model(videos)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            
    return correct / total

In [26]:
best_val_acc = 0

for epoch in range(20):

    train_loss = train_epoch(model, train_loader)
    val_acc = evaluate(model, val_loader)

    scheduler.step(val_acc)

    print(f"Epoch {epoch+1}")
    print("Train Loss:", train_loss)
    print("Val Accuracy:", val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        if isinstance(model, nn.DataParallel):
            torch.save(model.module.state_dict(), "best_model.pth")
        else:
            torch.save(model.state_dict(), "best_model.pth")
        print("Best model saved.")


test_acc = evaluate(model, test_loader)
print("Test Accuracy:", test_acc)

100%|██████████| 346/346 [08:53<00:00,  1.54s/it]


Epoch 1
Train Loss: 0.8343706091144526
Val Accuracy: 0.7929292929292929
Best model saved.


100%|██████████| 346/346 [08:55<00:00,  1.55s/it]


Epoch 2
Train Loss: 0.6385240490209161
Val Accuracy: 0.7525252525252525


100%|██████████| 346/346 [08:55<00:00,  1.55s/it]


...

Epoch 18
Train Loss: 0.04516078025772845
Val Accuracy: 0.8005050505050505


100%|██████████| 346/346 [08:56<00:00,  1.55s/it]


Epoch 19
Train Loss: 0.05249083729096112
Val Accuracy: 0.8207070707070707


100%|██████████| 346/346 [08:55<00:00,  1.55s/it]


Epoch 20
Train Loss: 0.062099864649198776
Val Accuracy: 0.8131313131313131
Test Accuracy: 0.810126582278481
